In [6]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import xarray as xr
import earthaccess
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import io

# --- Processing Functions ---

def extract_transect(ds, var_name, target_lats, target_lons):
    """Extracts the nearest pixels to a defined transect line."""
    lat_vals = ds["latitude"].values
    lon_vals = ds["longitude"].values
    data_da = ds[var_name]

    rows, cols = [], []
    for lat, lon in zip(target_lats, target_lons):
        dist = np.abs(lat_vals - lat) + np.abs(lon_vals - lon)
        i, j = np.unravel_index(dist.argmin(), lat_vals.shape)
        rows.append(i)
        cols.append(j)

    # Load only the small subset into memory
    selection = data_da.isel(
        number_of_lines=xr.DataArray(rows, dims="points"), 
        pixels_per_line=xr.DataArray(cols, dims="points")
    ).load()
    
    selection = selection.assign_coords(
        longitude=("points", [lon_vals[r, c] for r, c in zip(rows, cols)]),
        latitude=("points", [lat_vals[r, c] for r, c in zip(rows, cols)])
    )
    return selection

def plot_spectral_transect(ds_transect, file_id, date_str, output_path):
    """Generates and saves the spectral variation plot."""
    lons = ds_transect.longitude.values
    lats = ds_transect.latitude.values
    dist_degrees = np.sqrt((lons - lons[0])**2 + (lats - lats[0])**2)
    
    norm = colors.Normalize(vmin=dist_degrees.min(), vmax=dist_degrees.max())
    cmap = plt.get_cmap('plasma') 
    
    fig, ax = plt.subplots(figsize=(12, 7))
    for i, p in enumerate(ds_transect.points):
        point_data = ds_transect.sel(points=p)
        color = cmap(norm(dist_degrees[i]))
        point_data.plot.line(ax=ax, x='wavelength_3d', marker='.', color=color, alpha=0.6, add_legend=False)
    
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label('Distance (Degrees °)', rotation=270, labelpad=15)
    
    ax.set_title(f"Spectral Signatures: {date_str}", fontsize=14)
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Surface Reflectance (rhos)")
    ax.grid(True, linestyle='--', alpha=0.3)
    
    plt.savefig(os.path.join(output_path, f"{file_id}.png"), dpi=300)
    plt.close(fig)

def plot_transect_map(ds_transect, event_name, date_str, output_path):
    """Generates and saves a map showing the transect location (once per event)."""
    lons = ds_transect.longitude.values
    lats = ds_transect.latitude.values
    dist_degrees = np.sqrt((lons - lons[0])**2 + (lats - lats[0])**2)

    fig = plt.figure(figsize=(12, 9))
    ax = plt.axes(projection=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor='#f5f5f5', edgecolor='dimgray')
    ax.add_feature(cfeature.OCEAN, facecolor='#e3f2fd')
    ax.add_feature(cfeature.COASTLINE, linewidth=1)
    
    ax.plot(lons, lats, color='black', linestyle='-', linewidth=1, alpha=0.3, transform=ccrs.PlateCarree())
    
    norm = colors.Normalize(vmin=dist_degrees.min(), vmax=dist_degrees.max())
    path = ax.scatter(lons, lats, c=dist_degrees, cmap='plasma', s=60, 
                      edgecolors='white', linewidth=0.5, transform=ccrs.PlateCarree(), zorder=2)

    ax.plot(lons[0], lats[0], marker='D', color='green', markersize=8, transform=ccrs.PlateCarree(), label='Start')
    ax.plot(lons[-1], lats[-1], marker='X', color='red', markersize=10, transform=ccrs.PlateCarree(), label='End')

    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
    gl.top_labels = gl.right_labels = False

    cbar = plt.colorbar(path, ax=ax, orientation='vertical', shrink=0.6, pad=0.08)
    cbar.set_label('Distance (Degrees °)', rotation=270, labelpad=15)

    margin = 0.5
    ax.set_extent([lons.min() - margin, lons.max() + margin, lats.min() - margin, lats.max() + margin])
    
    plt.title(f"{event_name}", fontsize=14)
    plt.legend(loc='lower left')
    
    plt.savefig(os.path.join(output_path, f"{event_name}_map.png"), dpi=300)
    plt.close(fig)

def run_event_analysis(event_name, date_range, transect_lats, transect_lons):
    """Main workflow to process an event with memory purging."""
    output_dir = os.path.join("output_reports", event_name)
    os.makedirs(output_dir, exist_ok=True)
    
    bbox = (min(transect_lons)-1, min(transect_lats)-1, max(transect_lons)+1, max(transect_lats)+1)
    
    print(f"\n>>> Starting analysis for: {event_name}")
    results = earthaccess.search_data(short_name='PACE_OCI_L2_SFREFL', temporal=date_range, bounding_box=bbox)
    
    if not results:
        print(f"No files found for {event_name}")
        return
        
    fileset = earthaccess.open(results)
    map_generated = False

    for file_obj in fileset:
        try:
            # Use chunks={} for lazy loading (Dask)
            dt = xr.open_datatree(file_obj, decode_timedelta=False, chunks={})
            ds = xr.merge(dt.to_dict().values())
            ds = ds.set_coords(("longitude", "latitude"))
            
            rhos_transect = extract_transect(ds, 'rhos', transect_lats, transect_lons)
            
            file_id = file_obj.full_name.split('.')[1]
            formatted_date = f"{file_id[:4]}-{file_id[4:6]}-{file_id[6:8]} {file_id[9:11]}:{file_id[11:13]}"
            
            # Map generated only once per event
            if not map_generated:
                plot_transect_map(rhos_transect, event_name, formatted_date, output_dir)
                map_generated = True
                print(f"Event map generated: {event_name}_map.png")
            
            # Spectral plot per file
            plot_spectral_transect(rhos_transect, file_id, formatted_date, output_dir)
            print(f"Processed spectrum for: {file_id}")
            
            # --- MEMORY PURGE (Per File) ---
            ds.close()
            rhos_transect.close()
            gc.collect()
            del ds, dt, rhos_transect
            
        except Exception as e:
            print(f"Error processing {file_obj.full_name}: {e}")
        
        finally:
            # Clear Matplotlib backend and force garbage collection
            plt.close('all') 
            gc.collect()

    gc.collect()
    print(f"Memory purged after event: {event_name}")

# --- Main Block ---

In [ ]:
if __name__ == "__main__":
    # 1. Load Event Data
    df_events = pd.read_csv('events.txt')

    # 2. Iterate through events
    for index, row in df_events.iterrows():
        # Generate transect points (20 points between start and end)
        lats = np.linspace(row['lat_start'], row['lat_end'], 20)
        lons = np.linspace(row['lon_start'], row['lon_end'], 20)
        
        run_event_analysis(
            event_name = row['event_id'],
            date_range = (row['date_ini'], row['date_end']),
            transect_lats = lats,
            transect_lons = lons
        )

    print("\n--- All events processed successfully ---")


>>> Starting analysis for: Sonda_campeche


QUEUEING TASKS | :   0%|          | 0/488 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/488 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/488 [00:00<?, ?it/s]

Event map generated: Sonda_campeche_map.png
Processed spectrum for: 20240323T184230
Processed spectrum for: 20240324T191734


In [8]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from scipy.integrate import trapezoid

def plot_spectral_transect(ds_transect, file_id, date_str, output_path):
    """
    Generates a dual-panel plot showing Raw vs Area-Normalized spectral signatures.
    Saves the file with '_norm' suffix.
    """
    # 1. Coordinates and distance calculation
    lons = ds_transect.longitude.values
    lats = ds_transect.latitude.values
    dist_degrees = np.sqrt((lons - lons[0])**2 + (lats - lats[0])**2)
    
    # 2. Setup normalization and colormap
    norm = colors.Normalize(vmin=dist_degrees.min(), vmax=dist_degrees.max())
    cmap = plt.get_cmap('plasma') 
    
    # Create figure with 2 subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={'wspace': 0.2})
    
    # 3. Plotting loop
    for i, p in enumerate(ds_transect.points):
        point_data = ds_transect.sel(points=p)
        wavelengths = point_data.wavelength_3d.values
        reflectance = point_data.values
        
        # Color based on distance
        color = cmap(norm(dist_degrees[i]))
        
        # Plot A: Raw Data
        ax1.plot(wavelengths, reflectance, color=color, alpha=0.5, linewidth=0.8)
        
        # Plot B: Normalized Data
        mask = ~np.isnan(reflectance)
        if np.any(mask):
            # Calculate area using trapezoidal rule
            area = trapezoid(reflectance[mask], wavelengths[mask])
            if area > 0:
                normalized_reflectance = reflectance / area
                ax2.plot(wavelengths, normalized_reflectance, color=color, alpha=0.5, linewidth=0.8)

    # 4. Colorbar (Shared for both plots)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=[ax1, ax2], pad=0.02)
    cbar.set_label('Distance from Start (Degrees °)', rotation=270, labelpad=15)
    
    # 5. Formatting Subplot 1 (Raw)
    ax1.set_title(f"Raw Surface Reflectance\n{date_str}", fontsize=12, fontweight='bold')
    ax1.set_xlabel("Wavelength (nm)")
    ax1.set_ylabel("Reflectance (rhos)")
    ax1.grid(True, linestyle='--', alpha=0.3)
    ax1.axhline(0, color='black', linewidth=0.8, alpha=0.5)

    # 6. Formatting Subplot 2 (Normalized)
    ax2.set_title(f"Area-Normalized Reflectance\n(Integral = 1)", fontsize=12, fontweight='bold')
    ax2.set_xlabel("Wavelength (nm)")
    ax2.set_ylabel("Normalized Reflectance")
    ax2.grid(True, linestyle='--', alpha=0.3)
    ax2.axhline(0, color='black', linewidth=0.8, alpha=0.5)

    # 7. Save and close with 'norm' in filename
    output_filename = os.path.join(output_path, f"{file_id}_norm.png")
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.close(fig)
    
    return output_filename